In [1]:
import pandas as pd
import numpy as np
import jenkspy
from scipy.stats import kstest

import plotly.express as px
import plotly.graph_objects as go

import utility_functions as uf

In [2]:
path = "data/"

In [42]:
df_country_subfield = pd.read_csv(path+"df_country_subfield.csv", index_col="country")
# df_country_subfield = pd.read_csv(path+"df_country_subfield_authors.csv", index_col="country")

In [43]:
df_country_subfield

,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE,1.0,3.0,1.0,3.0,1.0,6.0,NaN,NaN,NaN,13.0,...,NaN,4.0,2.0,NaN,NaN,NaN,2.0,NaN,NaN,2.0
AF,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,...,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AL,NaN,NaN,NaN,NaN,3.0,2.0,1.0,NaN,NaN,2.0,...,NaN,2.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,NaN,2.0,NaN,1.0,NaN,1.0,NaN,NaN,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN
YE,NaN,3.0,2.0,1.0,1.0,3.0,NaN,NaN,1.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
ZA,30.0,51.0,52.0,16.0,214.0,77.0,17.0,NaN,66.0,290.0,...,6.0,27.0,8.0,2.0,NaN,16.0,16.0,1.0,14.0,13.0


In [27]:
df_country_subfield_norm = (
    df_country_subfield
    .div(df_country_subfield.sum(axis=1), axis=0)
)

In [73]:
df_country_stats = (
    df_country_subfield
    .div(df_country_subfield.sum(axis=1), axis=0)
    .assign(
        entropy=lambda df: -df.mul(np.log(df)).sum(axis=1),
        norm_entropy=lambda df: df.entropy / np.log(len(df.columns) - 1),
        total_articles=df_country_subfield.sum(axis=1),
        log_total_articles=lambda df: np.log(df.total_articles),
        gini=df_country_subfield.apply(uf.gini, axis=1)
    )
    .merge(uf.df_country[["alpha-2", "region"]], left_index=True, right_on="alpha-2", how="left")
    .rename(columns={"alpha-2": "country"})
    .set_index("country")
    .sort_values('total_articles', ascending=False)
    [["region", "entropy", "norm_entropy", "total_articles", "log_total_articles", "gini"]]
)

In [74]:
df_country_stats

,region,entropy,norm_entropy,total_articles,log_total_articles,gini
country,,,,,,
US,Americas,4.831009,0.873690,590234.0,13.288274,0.613181
CN,Asia,4.552830,0.823382,566541.0,13.247305,0.702698
GB,Europe,4.839565,0.875238,156317.0,11.959641,0.613604
DE,Europe,4.745337,0.858197,134641.0,11.810367,0.645080
FR,Europe,4.747294,0.858551,121503.0,11.707694,0.642908
...,...,...,...,...,...,...
GQ,Africa,0.693147,0.125356,2.0,0.693147,0.000000
SX,Americas,-0.000000,-0.000000,1.0,0.000000,0.000000
SB,Oceania,-0.000000,-0.000000,1.0,0.000000,0.000000


In [75]:
# df_country_stats.to_csv(path+"df_country_stats.csv")

# Size distribution

In [76]:
px.histogram(df_country_stats, x="log_total_articles", title="Number of Articles log-distribution")

In [77]:
# Example data
data = df_country_stats.log_total_articles.values

# KS test against standard normal
stat, p_value = kstest((data - data.mean()) / data.std(), 'norm')

print("KS statistic:", stat)
print("p-value:", p_value)
if p_value > 0.05:
    print("Normal distribution hypothesis is accepted")
else:
    print("Normal distribution hypothesis is rejected")

KS statistic: 0.054378986693755293
p-value: 0.5158849986810807
Normal distribution hypothesis is accepted


In [82]:
df_plot = (
    df_country_stats
    .reset_index()
    .sort_values("total_articles")
    .assign(total_cumsum=lambda df: df.total_articles.cumsum())
)
px.scatter(df_plot, x="country", y="total_cumsum", color="region", category_orders={"country": df_plot["country"].tolist()},
           title="Lorenz curve (country productivity)")

# Heterogeneity statistics

In [83]:
df_country_stats

,region,entropy,norm_entropy,total_articles,log_total_articles,gini
country,,,,,,
US,Americas,4.831009,0.873690,590234.0,13.288274,0.613181
CN,Asia,4.552830,0.823382,566541.0,13.247305,0.702698
GB,Europe,4.839565,0.875238,156317.0,11.959641,0.613604
DE,Europe,4.745337,0.858197,134641.0,11.810367,0.645080
FR,Europe,4.747294,0.858551,121503.0,11.707694,0.642908
...,...,...,...,...,...,...
GQ,Africa,0.693147,0.125356,2.0,0.693147,0.000000
SX,Americas,-0.000000,-0.000000,1.0,0.000000,0.000000
SB,Oceania,-0.000000,-0.000000,1.0,0.000000,0.000000


In [104]:
fig = px.bar(df_country_stats.reset_index(),
             x="norm_entropy",
             y="country",
             title="Normalized entropy sorted by number of articles",
             # color="region",
             category_orders={"country": df_plot["country"].tolist()})
fig.update_layout(height=800)

In [103]:
fig = px.bar(df_country_stats.reset_index(),
             x="gini",
             y="country",
             title="Gini sorted by number of articles",
             # color="region",
             category_orders={"country": df_plot["country"].tolist()})
fig.update_layout(height=800)

In [90]:
(
    df_country_stats
    # .query("gini > 0")
    # .query("total_articles > 400")
    .sort_values("gini", ascending=False)
    .merge(uf.df_country[["alpha-2", "name", "sub-region"]], left_index=True, right_on="alpha-2", how="left")
    .rename(columns={"alpha-2": "country"})
).head(10)

,region,entropy,norm_entropy,total_articles,log_total_articles,gini,country,name,sub-region
242.0,Americas,2.073851,0.375057,415.0,6.028279,0.789673,VG,Virgin Islands (British),Latin America and the Caribbean
183.0,Europe,4.349795,0.786663,34692.0,10.454264,0.740803,RU,Russian Federation,Eastern Europe
232.0,Europe,4.247213,0.768111,6240.0,8.738735,0.727324,UA,Ukraine,Eastern Europe
200.0,Asia,4.341850,0.785226,13430.0,9.505246,0.719332,SG,Singapore,South-eastern Asia
104.0,Asia,4.398014,0.795383,8942.0,9.098515,0.712656,ID,Indonesia,South-eastern Asia
217.0,NaN,4.413861,0.798249,33662.0,10.424125,0.703359,TW,"Taiwan, Province of China",NaN
45.0,Asia,4.552830,0.823382,566541.0,13.247305,0.702698,CN,China,Eastern Asia
112.0,Asia,4.525952,0.818521,113323.0,11.637997,0.699943,JP,Japan,Eastern Asia
119.0,Asia,4.473228,0.808986,52756.0,10.873433,0.691092,KR,"Korea, Republic of",Eastern Asia
103.0,Asia,4.512706,0.816125,57763.0,10.964104,0.687919,IN,India,Southern Asia


In [105]:
country = "VG"
px.bar(df_country_subfield.loc[country].to_frame(), title = uf.get_country_info(country).iloc[0, 0])

In [93]:
uf.get_subfield_info(3600)

,subfield_id,subfield_name,field_name,domain_name
311,3600,General Health Professions,Health Professions,Health Sciences


In [94]:
countries = ["RU", "BY", "UA"]
countries = ["GB", "CA", "IN", "ZA"]


fig = go.Figure()

for country in countries:
    fig.add_bar(
        x=df_country_subfield_norm.columns,
        y=df_country_subfield_norm.loc[country],
        name=country
    )

fig.update_layout(
    barmode="group",
    title=None,
    xaxis_title=None,
    yaxis_title=None
)

fig.show()

In [106]:
countries = ["VG", "RU", "BY", "UA", "GB", "CA", "IN", "ZA"]

fig = go.Figure()

for country in countries:
    y_values = (
        df_country_subfield
        .loc[country]
        .fillna(0)
        .to_frame()
        .sort_values(country)
        .cumsum()
        .div(
            df_country_subfield
            .loc[country]
            .fillna(0)
            .sum()
        )
    ).values.ravel()
    fig.add_trace(
        go.Scatter(
        x=np.arange(y_values.shape[0]),
        y=y_values,
        mode="lines",
        name=uf.get_country_info(country).iloc[0, 0] + " " + str(df_country_stats.loc[country].loc["gini"].round(2)),)
    )

fig.update_layout(title="Lorenz curve (articles distribution by sub-fields inequality)")

fig.show()

# Fisher breaks in heterogeneity statistics

In [107]:
breaks_gini = jenkspy.jenks_breaks(df_country_stats.gini, 7)

fig = go.Figure()

fig.add_trace(
    go.Histogram(x=df_country_stats.gini, nbinsx=100)
)
for x in breaks_gini:
    fig.add_vline(
        x=x,
        line=dict(color="red", width=2, dash="dash"),  # style of the line
        annotation_text=f"{x:.2f}",  # optional label
        annotation_position="top right"
    )
fig.update_layout(title="Gini histogram with Fisher breakpoints")
fig.show()

In [117]:
px.scatter(
    df_country_stats.reset_index(),
    x="log_total_articles",
    y="gini",
    title="Gini vs number of articles",
    color="region",
    hover_name="country",
    hover_data=["region", "total_articles", "gini"],
)

In [108]:
breaks_norm_entropy = jenkspy.jenks_breaks(df_country_stats.norm_entropy, 4)

fig = go.Figure()

fig.add_trace(
    go.Histogram(x=df_country_stats.norm_entropy, nbinsx=100)
)
for x in breaks_norm_entropy:
    fig.add_vline(
        x=x,
        line=dict(color="red", width=2, dash="dash"),  # style of the line
        annotation_text=f"{x:.2f}",  # optional label
        annotation_position="top right"
    )
fig.update_layout(title="Entropy histogram with Fisher breakpoints")
fig.show()